# Inspect and Audit Daily Ticker Context

This notebook audits `data/ticker_daily_context.parquet`. It independently recalculates every price, market, earnings, and fundamental field from the source datasets, then reports any mismatch. It does not modify the context table or source data.

This verifies the daily state is time-safe as of `price_data_as_of`. A later article-packet builder must separately enforce `price_data_as_of < published_at`.

In [10]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / '.env').exists() and (path / 'data').is_dir()
)
CONTEXT_PATH = PROJECT_ROOT / 'data' / 'ticker_daily_context.parquet'

load_dotenv(PROJECT_ROOT / '.env')
HF_TOKEN = os.getenv('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN is missing. Add it to the local .env file.'

api = HfApi(token=HF_TOKEN)
REPOS = {
    'prices': 'cookekieran/mag7_prices',
    'market': 'cookekieran/us_market_prices',
    'fundamentals': 'cookekieran/mag7-fundamentals',
}
REVISIONS = {name: api.repo_info(repo_id=repo, repo_type='dataset').sha for name, repo in REPOS.items()}
display(pd.Series(REVISIONS, name='revision').to_frame())

,revision
prices,e7cb2ea2a058da799c85718c28b77c6a9007ecab
market,0d61dd63fd98b05b678d8f3447190811a3294f66
fundamentals,de97de7a8efc130a258fb295a06520f9a8fed30f


## Load the context table and source data

In [11]:
def load_parquet(repo_name, filename):
    path = hf_hub_download(
        repo_id=REPOS[repo_name],
        repo_type='dataset',
        filename=filename,
        revision=REVISIONS[repo_name],
        token=HF_TOKEN,
    )
    return pd.read_parquet(path)

context = pd.read_parquet(CONTEXT_PATH)
context['price_data_as_of'] = pd.to_datetime(context['price_data_as_of'])

prices = load_parquet('prices', 'mag7_daily_prices.parquet')
market = load_parquet('market', 'sp500_daily_prices.parquet')
fundamentals = load_parquet('fundamentals', 'data/model_ready_fundamentals_wide.parquet')
earnings = load_parquet('fundamentals', 'data/earnings_dates_quarterly.parquet')

print(f'Context rows: {len(context):,}')
display(context.head())

Context rows: 5,985


,ticker,price_data_as_of,stock_return_1d,stock_return_5d,stock_return_20d,stock_return_45d,stock_return_60d,stock_return_90d,stock_return_120d,stock_volatility_20d,...,latest_operating_income,cash_and_short_term_investments,total_debt,revenue_growth_yoy,gross_margin,operating_margin,operating_margin_change_yoy,r_and_d_intensity,cash_to_debt,debt_to_assets
0,AAPL,2023-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,2023-01-04,0.010314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL,2023-01-05,-0.010605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL,2023-01-06,0.036794,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL,2023-01-09,0.004089,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
context.columns

Index(['ticker', 'price_data_as_of', 'stock_return_1d', 'stock_return_5d',
       'stock_return_20d', 'stock_return_45d', 'stock_return_60d',
       'stock_return_90d', 'stock_return_120d', 'stock_volatility_20d',
       'stock_volatility_60d', 'stock_volatility_90d', 'relative_volume_20d',
       'relative_volume_60d', 'relative_volume_90d', 'distance_from_20d_high',
       'distance_from_60d_high', 'distance_from_90d_high',
       'distance_from_20d_low', 'distance_from_60d_low',
       'distance_from_90d_low', 'distance_from_20d_moving_average',
       'sp500_return_1d', 'sp500_return_5d', 'sp500_return_20d',
       'sp500_return_45d', 'sp500_return_60d', 'sp500_return_90d',
       'sp500_return_120d', 'stock_minus_sp500_return_1d',
       'stock_minus_sp500_return_5d', 'stock_minus_sp500_return_20d',
       'stock_minus_sp500_return_45d', 'stock_minus_sp500_return_60d',
       'stock_minus_sp500_return_90d', 'stock_minus_sp500_return_120d',
       'days_since_last_earnings_report',

## Coverage and missing values

In [12]:
coverage = (
    context.groupby('ticker')
    .agg(first_date=('price_data_as_of', 'min'), last_date=('price_data_as_of', 'max'), rows=('ticker', 'size'))
    .reset_index()
)
display(coverage)
display(context.isna().sum().rename('missing_values').to_frame())

,ticker,first_date,last_date,rows
0,AAPL,2023-01-03,2026-06-01,855
1,AMZN,2023-01-03,2026-06-01,855
2,GOOGL,2023-01-03,2026-06-01,855
3,META,2023-01-03,2026-06-01,855
4,MSFT,2023-01-03,2026-06-01,855
5,NVDA,2023-01-03,2026-06-01,855
6,TSLA,2023-01-03,2026-06-01,855


,missing_values
ticker,0
price_data_as_of,0
stock_return_1d,7
stock_return_5d,35
stock_return_20d,140
stock_return_45d,315
stock_return_60d,420
stock_return_90d,630
stock_return_120d,840
stock_volatility_20d,140


## Recalculate price and market features from source prices

In [13]:
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values(['ticker', 'date'])
RETURN_WINDOWS = (1, 5, 20, 45, 60, 90, 120)
STATE_WINDOWS = (20, 60, 90)

for days in RETURN_WINDOWS:
    prices[f'stock_return_{days}d'] = prices.groupby('ticker')['adj_close'].pct_change(days)

for days in STATE_WINDOWS:
    prices[f'stock_volatility_{days}d'] = prices.groupby('ticker')['stock_return_1d'].transform(lambda x: x.rolling(days).std())
    prices[f'relative_volume_{days}d'] = prices.groupby('ticker')['volume'].transform(lambda x: x / x.rolling(days).mean().shift(1))
    prices[f'distance_from_{days}d_high'] = prices.groupby('ticker')['adj_close'].transform(lambda x: x / x.rolling(days).max() - 1)
    prices[f'distance_from_{days}d_low'] = prices.groupby('ticker')['adj_close'].transform(lambda x: x / x.rolling(days).min() - 1)
prices['distance_from_20d_moving_average'] = prices.groupby('ticker')['adj_close'].transform(lambda x: x / x.rolling(20).mean() - 1)

market['date'] = pd.to_datetime(market['date'])
market = market.sort_values('date')
for days in RETURN_WINDOWS:
    market[f'sp500_return_{days}d'] = market['adj_close'].pct_change(days)

expected_price = prices.merge(
    market[['date', *[f'sp500_return_{days}d' for days in RETURN_WINDOWS]]],
    on='date',
    how='left',
)
for days in RETURN_WINDOWS:
    expected_price[f'stock_minus_sp500_return_{days}d'] = expected_price[f'stock_return_{days}d'] - expected_price[f'sp500_return_{days}d']

expected_price = expected_price.rename(columns={'date': 'price_data_as_of'})

## Audit price and market provenance

Every saved price-state row must map to a real source trading day, and every saved metric must equal its independently recalculated value.

In [14]:
price_columns = [
    *[f'stock_return_{days}d' for days in RETURN_WINDOWS],
    *[f'stock_volatility_{days}d' for days in STATE_WINDOWS],
    *[f'relative_volume_{days}d' for days in STATE_WINDOWS],
    *[f'distance_from_{days}d_high' for days in STATE_WINDOWS],
    *[f'distance_from_{days}d_low' for days in STATE_WINDOWS],
    'distance_from_20d_moving_average',
    *[f'sp500_return_{days}d' for days in RETURN_WINDOWS],
    *[f'stock_minus_sp500_return_{days}d' for days in RETURN_WINDOWS],
]

price_audit = context.merge(
    expected_price[['ticker', 'price_data_as_of', *price_columns]],
    on=['ticker', 'price_data_as_of'],
    how='left',
    suffixes=('_saved', '_expected'),
)

price_results = []
for column in price_columns:
    matches = np.isclose(
        price_audit[f'{column}_saved'],
        price_audit[f'{column}_expected'],
        equal_nan=True,
    )
    price_results.append({'field': column, 'mismatched_rows': int((~matches).sum())})

source_date_rows = price_audit['stock_return_1d_expected'].notna() | price_audit['stock_return_1d_saved'].isna()
price_results.append({'field': 'source_price_date_exists', 'mismatched_rows': int((~source_date_rows).sum())})
display(pd.DataFrame(price_results))

,field,mismatched_rows
0,stock_return_1d,0
1,stock_return_5d,0
2,stock_return_20d,0
3,stock_return_45d,0
4,stock_return_60d,0
5,stock_return_90d,0
6,stock_return_120d,0
7,stock_volatility_20d,0
8,stock_volatility_60d,0
9,stock_volatility_90d,0


## Recalculate earnings and fundamental state

A report is only available from the next calendar day. The audit checks that no context row uses a report dated on or after its `price_data_as_of` date.

In [15]:
fundamentals['fiscal_period_end'] = pd.to_datetime(fundamentals['fiscal_period_end'])
earnings['fiscal_period_end'] = pd.to_datetime(earnings['fiscal_period_end'])
earnings['reported_date'] = pd.to_datetime(earnings['reported_date'])

financials = earnings.merge(fundamentals, on=['ticker', 'fiscal_period_end'], how='left').sort_values(['ticker', 'fiscal_period_end'])
revenue = 'fundamental_income_statement_totalRevenue'
operating_income = 'fundamental_income_statement_operatingIncome'
gross_profit = 'fundamental_income_statement_grossProfit'
research_and_development = 'fundamental_income_statement_researchAndDevelopment'
cash = 'fundamental_balance_sheet_cashAndShortTermInvestments'
debt = 'fundamental_balance_sheet_shortLongTermDebtTotal'
assets = 'fundamental_balance_sheet_totalAssets'

financials['revenue_growth_yoy'] = financials.groupby('ticker')[revenue].pct_change(4, fill_method=None)
financials['gross_margin'] = financials[gross_profit] / financials[revenue]
financials['operating_margin'] = financials[operating_income] / financials[revenue]
financials['operating_margin_change_yoy'] = financials.groupby('ticker')['operating_margin'].diff(4)
financials['r_and_d_intensity'] = financials[research_and_development] / financials[revenue]
financials['cash_to_debt'] = financials[cash] / financials[debt].where(financials[debt] != 0)
financials['debt_to_assets'] = financials[debt] / financials[assets]
financials['available_from'] = financials['reported_date'] + pd.Timedelta(days=1)

financial_columns = [
    'ticker', 'available_from', 'reported_date', 'reportedEPS', 'estimatedEPS',
    'surprisePercentage', 'reportTime', revenue, operating_income, cash, debt,
    'revenue_growth_yoy', 'gross_margin', 'operating_margin',
    'operating_margin_change_yoy', 'r_and_d_intensity', 'cash_to_debt', 'debt_to_assets',
]

expected_financials = pd.merge_asof(
    context[['ticker', 'price_data_as_of']].sort_values(['price_data_as_of', 'ticker']),
    financials[financial_columns].sort_values(['available_from', 'ticker']),
    left_on='price_data_as_of',
    right_on='available_from',
    by='ticker',
    direction='backward',
)
expected_financials['days_since_last_earnings_report'] = (
    expected_financials['price_data_as_of'] - expected_financials['reported_date']
).dt.days

## Audit financial provenance and leakage

In [16]:
financial_mapping = {
    'days_since_last_earnings_report': 'days_since_last_earnings_report',
    'latest_reported_eps': 'reportedEPS',
    'latest_estimated_eps': 'estimatedEPS',
    'latest_eps_surprise_percentage': 'surprisePercentage',
    'latest_revenue': revenue,
    'latest_operating_income': operating_income,
    'cash_and_short_term_investments': cash,
    'total_debt': debt,
    'revenue_growth_yoy': 'revenue_growth_yoy',
    'gross_margin': 'gross_margin',
    'operating_margin': 'operating_margin',
    'operating_margin_change_yoy': 'operating_margin_change_yoy',
    'r_and_d_intensity': 'r_and_d_intensity',
    'cash_to_debt': 'cash_to_debt',
    'debt_to_assets': 'debt_to_assets',
}

expected_names = ['reported_date', 'reportTime', *financial_mapping.values()]
expected_names = list(dict.fromkeys(expected_names))
expected_financials = expected_financials.rename(
    columns={name: f'{name}_expected' for name in expected_names}
)

financial_audit = context.merge(
    expected_financials[['ticker', 'price_data_as_of', *[f'{name}_expected' for name in expected_names]]],
    on=['ticker', 'price_data_as_of'],
    how='left',
    suffixes=('_saved', '_expected'),
)

financial_results = []
for saved_name, expected_name in financial_mapping.items():
    matches = np.isclose(
        financial_audit[saved_name],
        financial_audit[f'{expected_name}_expected'],
        equal_nan=True,
    )
    financial_results.append({'field': saved_name, 'mismatched_rows': int((~matches).sum())})

report_time_matches = (
    financial_audit['latest_report_time'].fillna('')
    == financial_audit['reportTime_expected'].fillna('')
)
financial_results.append({'field': 'latest_report_time', 'mismatched_rows': int((~report_time_matches).sum())})

future_reports = (financial_audit['reported_date_expected'] >= financial_audit['price_data_as_of']).fillna(False)
financial_results.append({'field': 'future_or_same_day_report_used', 'mismatched_rows': int(future_reports.sum())})
display(pd.DataFrame(financial_results))

,field,mismatched_rows
0,days_since_last_earnings_report,0
1,latest_reported_eps,0
2,latest_estimated_eps,0
3,latest_eps_surprise_percentage,0
4,latest_revenue,0
5,latest_operating_income,0
6,cash_and_short_term_investments,0
7,total_debt,0
8,revenue_growth_yoy,0
9,gross_margin,0


## Audit conclusion

A clean daily-context audit has zero mismatches in both audit tables and zero `future_or_same_day_report_used` rows. When article packets are built, add the final mandatory test: every packet must satisfy `price_data_as_of < published_at` before it is sent to the LLM.